In [ ]:
%load_ext autoreload
%autoreload 2
import os
import time
import random
import logging
import itertools
import copy
import numpy as np
import multiprocessing
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from models.resnet import load_model, CriticModel, get_model, SimpleClassifier
from utils import AverageMeter, save_checkpoint, accuracy
from dataset.jetClass_datasetIterable import JetDatasetIterable
import math
from datetime import datetime
import importlib
from tqdm import tqdm
import dataset.jetClass_datasetIterable
from models.parT import ParticleTransformerWrapper
from models.parT import ParTCritic, ParTCriticMass
import tqdm


In [3]:
import dataset

In [4]:
in_dataset = 'jetClass'

In [5]:
if in_dataset == "waterbird":
    dataset_class = WaterbirdDataset
elif in_dataset == "cmnist":
    dataset_class = ColoredMNIST
elif in_dataset == 'jetClass':
    dataset_class = JetDatasetIterable
else:
    raise ValueError(f"in_dataset not supported: {args.in_dataset}.")

In [6]:
data_path = '/Users/anrunw/Downloads/nuisance-aware-ood-detection-master 2/training'

In [7]:


train_loader = dataset.jetClass_datasetIterable.get_JetClass_iterable_dataloader(data_path, batch_size=64)
val_loader = dataset.jetClass_datasetIterable.get_JetClass_iterable_dataloader(data_path, batch_size=64)


In [8]:
num_classes = 10

In [9]:
    def __getitem__(self, index):
        x_particles, x_jets, x_points, x_mask = self.data_list[index]
        label = self.labels[index]
        nuisance = self.nuisances[index]
        return x_particles, x_jets, x_points, x_mask, label, nuisance

    def __len__(self):
        return len(self.labels)

    def get_label_prior(self):
        num_classes = np.max(self.labels) + 1
        label_counts = np.bincount(self.labels, minlength=num_classes)
        total_samples = len(self.labels)
        label_prior = {i: count / total_samples for i, count in enumerate(label_counts)}
        return label_prior

    def get_nuisance_prior(self):
        num_examples = len(self.nuisances)
        nuisance_counts = Counter(self.nuisances)
        nuisance_prior = {k: v / num_examples for k, v in nuisance_counts.items()}
        print(nuisance_prior)
        return nuisance_prior

In [10]:
cfg = dict(
    input_dim=17,
    num_classes=10,
    # network configurations
    pair_input_dim=4,
    use_pre_activation_pair=False,
    embed_dims=[128, 512, 128],
    pair_embed_dims=[64, 64, 64],
    num_heads=8,
    num_layers=8,
    num_cls_layers=2,
    block_params=None,
    cls_block_params={'dropout': 0, 'attn_dropout': 0, 'activation_dropout': 0},        fc_params=[],
    activation='gelu',
    # misc
    trim=True,
    for_inference=False,
)


In [11]:
device = 'cpu'

In [ ]:
base_model = ParticleTransformerWrapper(**cfg)
state_dict = torch.load('/Users/anrunw/EfficientTransformer/efficient_particle_transformer/networks/ParT_full.pt', map_location=torch.device(device))
base_model.load_state_dict(state_dict)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-4)


In [19]:
critic = ParTCritic(128)
criticLoss = nn.BCELoss().to(device)


In [15]:
criticOpt = torch.optim.Adam(base_model.parameters(), lr=1e-4)


### Training Critic Only


In [ ]:
from tqdm import tqdm
import numpy as np
import torch

# Training settings
epochs = 100
trainloss = np.zeros(epochs)
valloss = np.zeros(epochs)

# Training loop (only training the critic)
for epoch in tqdm(range(epochs)):
    batchNum = 0

    for batch in train_loader:
        # Ensure all tensors in the batch are on the correct device and dtype
        x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
        y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

        # Forward pass through the base model (no gradients needed)
        with torch.no_grad():
            y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

        # Forward pass through the critic
        criticOut = critic(activations.float(), y_one_hot, nuisance.view(-1, 1).float())
        critic_labels = torch.ones_like(criticOut, dtype=torch.float32)  # Match shape with `criticOut`

        # Compute critic loss
        critic_loss = criticLoss(criticOut.squeeze(), critic_labels.squeeze())

        # Backward pass and optimizer step for the critic
        criticOpt.zero_grad()
        critic_loss.backward()
        criticOpt.step()

        # Store the training loss for this epoch (critic loss only)
        trainloss[epoch] = critic_loss.item()
        print(f'Batch Number {batchNum} Critic Loss = {critic_loss:.4f}')

        batchNum += 1

    # Validation step
    ylosstot = []
    for batch in val_loader:
        with torch.no_grad():
            x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
            y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

            # Forward pass through the base model to get activations
            y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

            # Forward pass through the critic
            criticOut = critic(activations.float(), y_one_hot, nuisance.view(-1, 1).float())
            critic_labels = torch.ones_like(criticOut, dtype=torch.float32)

            # Compute validation critic loss
            yloss = criticLoss(criticOut.squeeze(), critic_labels.squeeze())
            ylosstot.append(yloss.item())

    valloss[epoch] = np.mean(ylosstot)

    # Print training and validation loss for this epoch
    print(f'Epoch {epoch + 1} Critic Train Loss: {trainloss[epoch]:.4f}')
    print(f'        Critic Val Loss: {valloss[epoch]:.4f}')


### Training critic and model simultaneously

In [ ]:
from tqdm import tqdm
import numpy as np
import torch

# Training settings
epochs = 100
trainloss = np.zeros(epochs)
valloss = np.zeros(epochs)

# Training loop
for epoch in tqdm(range(epochs)):
    batchNum = 0

    for batch in train_loader:
        # Toggle critic training every 10 batches
        if batchNum % 1 == 0 and batchNum != 0:
            train_critic = True
        else:
            train_critic = False

        # Ensure all tensors in the batch are on the correct device and dtype
        x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
        y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

        # Forward pass through the base model
        y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

        if train_critic:

            # Forward pass through the critic
            criticOut = critic(activations.float(), y_one_hot, nuisance.view(-1, 1).float())
            critic_labels = torch.ones_like(criticOut, dtype=torch.float32)  # Match shape with `criticOut`

            # Compute critic loss
            critic_loss = criticLoss(criticOut.squeeze(), critic_labels.squeeze())

            # Backward pass and optimizer step for the critic
            #criticOpt.zero_grad()
            #critic_loss.backward(retain_graph=True)  # Retain the graph for the base model's backward pass
            #criticOpt.step()
        else:
            critic_loss = torch.zeros(1, device=device).sum()  # Zero critic loss when not training critic

        # Compute combined loss
        loss = criterion(y_pred, y_one_hot) + 1.5 * critic_loss

        # Backward pass and optimizer step for the base model
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store the training loss for this epoch
        trainloss[epoch] = loss.item()
        print(f'Batch Number {batchNum} Total Loss = {loss:.4f} Loss = {criterion(y_pred, y_one_hot):.4f} Critic Loss = {critic_loss:.4f} ')

        batchNum += 1

    # Validation step
    ylosstot = []
    for batch in val_loader:
        with torch.no_grad():
            x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
            y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

            # Forward pass
            y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

            # Compute validation loss
            yloss = criterion(y_pred, y_one_hot)
            ylosstot.append(yloss.item())

    valloss[epoch] = np.mean(ylosstot)

    # Print training and validation loss for this epoch
    print(f'Epoch {epoch + 1} Train Loss: {trainloss[epoch]:.4f}')
    print(f'        Val Loss: {valloss[epoch]:.4f}')


### When predicting Mass

In [57]:
critic = ParTCriticMass(128)
criticLoss = nn.MSELoss().to(device)

In [ ]:
from tqdm import tqdm
import numpy as np
import torch

# Training settings
epochs = 100
trainloss = np.zeros(epochs)
valloss = np.zeros(epochs)

# Training loop
for epoch in tqdm(range(epochs)):
    batchNum = 0

    for batch in train_loader:
        if batchNum % 1 == 0 and batchNum != 0:
            train_critic = True
        else:
            train_critic = False

        # Ensure all tensors in the batch are on the correct device and dtype
        x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
        y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

        # Forward pass through the base model
        y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

        if train_critic:

            # Forward pass through the critic
            criticOut = critic(activations.float())

            # Compute critic loss
            critic_loss = criticLoss(criticOut.squeeze(), nuisance)

            # Backward pass and optimizer step for the critic
            #criticOpt.zero_grad()
            #critic_loss.backward(retain_graph=True)  # Retain the graph for the base model's backward pass
            #criticOpt.step()
        else:
            critic_loss = torch.zeros(1, device=device).sum()  # Zero critic loss when not training critic

        # Compute combined loss
        loss = criterion(y_pred, y_one_hot) + critic_loss

        # Backward pass and optimizer step for the base model
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store the training loss for this epoch
        trainloss[epoch] = loss.item()
        print(f'Batch Number {batchNum} Total Loss = {loss:.4f} Loss = {criterion(y_pred, y_one_hot):.4f} Critic Loss = {critic_loss:.4f} ')

        batchNum += 1

    # Validation step
    ylosstot = []
    for batch in val_loader:
        with torch.no_grad():
            x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
            y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

            # Forward pass
            y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

            # Compute validation loss
            yloss = criterion(y_pred, y_one_hot)
            ylosstot.append(yloss.item())

    valloss[epoch] = np.mean(ylosstot)

    # Print training and validation loss for this epoch
    print(f'Epoch {epoch + 1} Train Loss: {trainloss[epoch]:.4f}')
    print(f'        Val Loss: {valloss[epoch]:.4f}')


### Alternate between each model

In [ ]:
from tqdm import tqdm
import numpy as np
import torch

# Training settings
epochs = 100
trainloss = np.zeros(epochs)
valloss = np.zeros(epochs)

# Training loop
for epoch in tqdm(range(epochs)):
    batchNum = 0

    # Alternate training every 10 epochs
    train_critic = (epoch // 10) % 2 == 0  # Train critic for 10 epochs, then switch

    for batch in train_loader:
        # Ensure all tensors in the batch are on the correct device and dtype
        x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
        y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

        # Forward pass through the base model
        y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

        if train_critic:
            # Freeze base model parameters
            for param in base_model.parameters():
                param.requires_grad = False
            for param in critic.parameters():
                param.requires_grad = True

            # Forward pass through the critic
            criticOut = critic(activations.float(), y_one_hot, nuisance.view(-1, 1).float())
            critic_labels = torch.ones_like(criticOut, dtype=torch.float32)

            # Compute critic loss
            critic_loss = criticLoss(criticOut.squeeze(), critic_labels.squeeze())

            # Backward pass and optimizer step for the critic
            criticOpt.zero_grad()
            critic_loss.backward()
            criticOpt.step()
        else:
            # Freeze critic parameters
            for param in base_model.parameters():
                param.requires_grad = True
            for param in critic.parameters():
                param.requires_grad = False

            # Compute base model loss
            critic_loss = torch.zeros(1, device=device).sum()  # Zero critic loss when not training critic

        # Compute total loss
        base_loss = criterion(y_pred, y_one_hot)
        loss = base_loss + critic_loss

        if not train_critic:  # Train the base model only
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Store the training loss for this epoch
        trainloss[epoch] = loss.item()
        print(f'Batch Number {batchNum} Total Loss = {loss:.4f}, Base Model Loss = {base_loss:.4f}, Critic Loss = {critic_loss:.4f}')

        batchNum += 1

    # Validation step
    ylosstot = []
    for batch in val_loader:
        with torch.no_grad():
            x_particles, x_jets, x_points, x_mask, y, nuisance = [b.to(device).float() for b in batch]
            y_one_hot = torch.nn.functional.one_hot(y.long(), num_classes=num_classes).float()

            # Forward pass
            y_pred, activations = base_model(x_points, x_particles, x_jets, x_mask)

            # Compute validation loss
            yloss = criterion(y_pred, y_one_hot)
            ylosstot.append(yloss.item())

    valloss[epoch] = np.mean(ylosstot)

    # Print training and validation loss for this epoch
    print(f'Epoch {epoch + 1} Total Train Loss: {trainloss[epoch]:.4f}, Validation Loss: {valloss[epoch]:.4f}')
